# m04_l04_tool_failures

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Google Colab only: fetches the free code companion, then run the rest.
import io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if "google.colab" in sys.modules and not os.path.isdir("shopassist"):
    try:
        data = io.BytesIO(urllib.request.urlopen(URL).read())
    except OSError:  # network blocks the download: upload the zip
        from google.colab import files
        data = next(iter(files.upload()))
    zipfile.ZipFile(data).extractall(".")
    os.chdir(next(d for d in os.listdir(".") if d.endswith("-companion")))
    sys.path.insert(0, os.getcwd())


## Tool failures

In [ ]:
import time

from shopassist.actions import request_address_change
from shopassist.agent import run_tool
from shopassist.messages import ToolCall
from shopassist.toolkit import guarded
from shopassist.tools import GET_ORDER, Tool

# 1. Wrong argument type: the model sent a number, not a string.
call = ToolCall("c1", "get_order", {"order_id": 1042})
print("bad args: ", run_tool(guarded(GET_ORDER), call).text)

# 2. A tool that hangs: stop waiting after 0.2 seconds.
slow = Tool("check_warehouse", "Slow lookup.",
            {"type": "object", "properties": {}}, lambda: time.sleep(1))
call = ToolCall("c2", "check_warehouse", {})
print("too slow: ", run_tool(guarded(slow, timeout=0.2), call).text)

# 3. The same change requested twice (a retry): applied once.
for _ in range(2):
    r = request_address_change("1042", "12 MG Road", "req-7")
    print("change:   ", "duplicate ignored" if r["duplicate"] else
          "recorded")
